# 🕸️ Lab 12: Graph-Structured Data
**BINF 4002 – Machine Learning for Health**

---
## Learning Objectives
1. Understand how data can be represented as graphs (nodes, edges, features, adjacency matrices)
2. Load and explore real molecular graph datasets used in drug discovery
3. Build a **baseline** by extracting hand-crafted graph features and applying classical ML ("tabularization")
4. Understand the **message-passing** framework that underpins Graph Neural Networks (GNNs)
5. Implement and train a GNN (Graph Convolutional Network & Graph Isomorphism Network) using PyTorch Geometric
6. Compare tabularization baselines with GNN approaches on a molecular property prediction task

### Why Graphs in Healthcare?
Graph-structured data is ubiquitous in biomedicine:
- **Molecules** are graphs of atoms (nodes) connected by bonds (edges) → drug discovery, toxicity prediction
- **Protein structures** are residue contact graphs → protein function prediction
- **Knowledge graphs** encode relationships between diseases, genes, and drugs → drug repurposing
- **Patient networks** connect patients by shared conditions or encounters → epidemiology

In this lab, we focus on **molecular property prediction** — given the graph structure of a molecule,
predict whether it has a particular biological property. This is a core task in computational drug
discovery and pharmacology.

### Dataset: BBBP (Blood-Brain Barrier Penetration)
We use the **BBBP** dataset from MoleculeNet, which contains ~2,000 molecules labeled by whether
they can penetrate the blood-brain barrier — a critical property for neurological drug design.
Each molecule is represented as a graph where atoms are nodes and bonds are edges.

## Set-up
### Install Dependencies

This lab requires **PyTorch** and **PyTorch Geometric (PyG)**, the leading library for
deep learning on graphs. Run the cell below to install everything needed.
(This may take 1–2 minutes on Colab.)

In [ ]:
# ── Install PyTorch Geometric and dependencies ──────────────────────────────
# This cell handles installation for Google Colab.
# If running locally, see: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import torch
    print(f"✅ PyTorch {torch.__version__} already installed")
except ImportError:
    install("torch")
    import torch
    print(f"✅ PyTorch {torch.__version__} installed")

try:
    import torch_geometric
    print(f"✅ PyTorch Geometric {torch_geometric.__version__} already installed")
except ImportError:
    install("torch_geometric")
    install("pyg_lib")
    install("torch_scatter")
    install("torch_sparse")
    install("torch_cluster")
    install("torch_spline_conv")
    import torch_geometric
    print(f"✅ PyTorch Geometric {torch_geometric.__version__} installed")

try:
    import rdkit
    print(f"✅ RDKit {rdkit.__version__} already installed")
except ImportError:
    install("rdkit")
    import rdkit
    print(f"✅ RDKit installed")

try:
    import sklearn
    print(f"✅ scikit-learn {sklearn.__version__} already installed")
except ImportError:
    install("scikit-learn")

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj, degree
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool, global_add_pool

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("✅ All imports successful")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

---
## Part 1 — What Is Graph-Structured Data?

A **graph** $G = (V, E)$ consists of:
- **Nodes** (vertices) $V = \{v_1, v_2, \ldots, v_n\}$ — the entities
- **Edges** $E \subseteq V \times V$ — the relationships between entities

Graphs can carry additional information:
- **Node features** $\mathbf{X} \in \mathbb{R}^{n \times d}$: a $d$-dimensional feature vector per node
- **Edge features** $\mathbf{E}_{ij}$: attributes on each edge (e.g., bond type)
- **Graph-level labels** $y$: a label for the entire graph (our prediction target)

### Molecules as Graphs
In a molecular graph:
- **Nodes** = atoms (carbon, oxygen, nitrogen, ...)
- **Edges** = chemical bonds (single, double, aromatic, ...)
- **Node features** = atom properties (atomic number, degree, charge, aromaticity, ...)
- **Edge features** = bond properties (bond type, stereochemistry, ...)
- **Graph label** = molecular property (e.g., does it cross the blood-brain barrier?)

We start by building a small graph by hand to understand the representation.

In [ ]:
# ── Build a toy molecular graph: ethanol (C₂H₅OH) ──────────────────────────
# We'll represent this manually, then visualize it.
# Ethanol heavy-atom graph (ignoring hydrogens): C-C-O

import networkx as nx

G_toy = nx.Graph()
G_toy.add_node(0, element='C', color='#404040')
G_toy.add_node(1, element='C', color='#404040')
G_toy.add_node(2, element='O', color='#e74c3c')

G_toy.add_edge(0, 1, bond='single')
G_toy.add_edge(1, 2, bond='single')

# Adjacency matrix
A = nx.adjacency_matrix(G_toy).toarray()
print("Adjacency matrix A:")
print(A)
print(f"\nNumber of nodes: {G_toy.number_of_nodes()}")
print(f"Number of edges: {G_toy.number_of_edges()}")
print(f"Node degrees: {dict(G_toy.degree())}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

pos = {0: (0, 0), 1: (1, 0), 2: (2, 0)}
colors = [G_toy.nodes[n]['color'] for n in G_toy.nodes()]
labels = {n: G_toy.nodes[n]['element'] for n in G_toy.nodes()}
nx.draw(G_toy, pos, ax=axes[0], with_labels=True, labels=labels,
        node_color=colors, node_size=800, font_size=16, font_color='white',
        edge_color='#888', width=3, font_weight='bold')
axes[0].set_title("Ethanol (heavy atoms)", fontsize=13)

im = axes[1].imshow(A, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
axes[1].set_xticklabels(['C₀', 'C₁', 'O₂']); axes[1].set_yticklabels(['C₀', 'C₁', 'O₂'])
axes[1].set_title("Adjacency Matrix", fontsize=13)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(A[i, j]), ha='center', va='center', fontsize=14,
                     color='white' if A[i, j] else 'black')

plt.tight_layout()
plt.show()

### 🤔 Reflection 1.1 — Graphs vs. Tabular Data

1. In tabular data, every sample has the same number of features in a fixed order.
   Why can't we directly represent a graph in a fixed-length feature vector?
   (Hint: think about molecules of different sizes.)

2. The adjacency matrix $A$ is symmetric for undirected graphs. What would it mean
   biologically if the molecular graph were *directed*?

3. If we have a molecule with $n$ atoms and $m$ bonds, what is the shape of the adjacency
   matrix? How does this compare to the number of features in tabular data? What
   challenges does this create for batching multiple molecules together?

4. Two molecules can have the same set of atoms but different structures (isomers).
   How does graph representation capture this distinction, while a simple atom-count
   feature vector would not?

In [ ]:
# ══ SOLUTION — Reflection 1.1 ═══════════════════════════════════════════════
# 1. Graphs have variable numbers of nodes and edges — a drug molecule might have 20
#    atoms while a protein fragment has 200. Tabular formats require a fixed number of
#    columns. We cannot simply stack adjacency matrices of different sizes into a matrix.
#    Moreover, graphs have no canonical node ordering, so the same graph can have many
#    different adjacency matrices (one per node permutation).
#
# 2. Directed edges would imply an asymmetric relationship between atoms — e.g.,
#    electron donation in a coordinate bond, or flow in a metabolic pathway. In most
#    molecular graphs, bonds are undirected (symmetric), but directed graphs are used
#    in reaction networks and regulatory pathways.
#
# 3. The adjacency matrix is n × n. For a 50-atom molecule, that's 2,500 entries (though
#    sparse — most atoms aren't bonded to most others). In tabular data, we might have
#    d features per sample. The challenge for batching is that different molecules have
#    different n, so we can't simply stack adjacency matrices. PyG handles this by
#    storing graphs in edge-index (COO) format and batching via block-diagonal adjacency.
#
# 4. Isomers like butane vs. isobutane have the same atom counts (C4H10) but different
#    connectivity. A graph captures which atom is bonded to which, while a simple atom
#    count vector {C:4, H:10} is identical for both. Graph isomorphism is precisely what
#    GNNs try to distinguish (the Weisfeiler-Leman test).
print("See comments above for solution.")

---
## Part 2 — Loading and Exploring Molecular Graph Data

We now load the **BBBP** (Blood-Brain Barrier Penetration) dataset from MoleculeNet.
PyTorch Geometric provides this dataset with molecules pre-converted to graph objects,
where each graph has:
- `x`: node feature matrix (atom features), shape `[num_atoms, num_atom_features]`
- `edge_index`: edge list in COO format, shape `[2, num_edges]`
- `edge_attr`: edge features (bond type, etc.)
- `y`: graph label (1 = penetrates BBB, 0 = does not)
- `smiles`: the SMILES string representation of the molecule

In [ ]:
# ── Load the BBBP dataset ─────────────────────────────────────────────────
dataset = MoleculeNet(root='/tmp/BBBP', name='BBBP')

print(f"Dataset: {dataset}")
print(f"Number of molecules: {len(dataset)}")
print(f"Number of node features: {dataset.num_node_features}")
print(f"Number of edge features: {dataset.num_edge_features}")
print(f"Number of classes: {dataset.num_classes}")

# Look at a single molecule
mol = dataset[0]
print(f"\n── First molecule ──")
print(f"  SMILES: {mol.smiles}")
print(f"  Number of atoms (nodes): {mol.x.shape[0]}")
print(f"  Node feature dim: {mol.x.shape[1]}")
print(f"  Number of bonds (edges): {mol.edge_index.shape[1] // 2}  (stored as {mol.edge_index.shape[1]} directed)")
print(f"  Label (BBB penetration): {mol.y.item()}")
print(f"  Node features (first 3 atoms):\n{mol.x[:3]}")
print(f"  Edge index (first 6 columns):\n{mol.edge_index[:, :6]}")

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────
num_atoms_list = []
num_edges_list = []
labels = []

for data in dataset:
    num_atoms_list.append(data.x.shape[0])
    num_edges_list.append(data.edge_index.shape[1] // 2)
    if data.y.dim() > 0:
        labels.append(data.y.squeeze().item())
    else:
        labels.append(data.y.item())

num_atoms_arr = np.array(num_atoms_list)
num_edges_arr = np.array(num_edges_list)
labels_arr = np.array(labels)

# Remove NaN labels if any
valid_mask = ~np.isnan(labels_arr)
print(f"Valid labels: {valid_mask.sum()} / {len(labels_arr)}")
labels_arr_clean = labels_arr[valid_mask]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(num_atoms_arr, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Number of Atoms')
axes[0].set_ylabel('Count')
axes[0].set_title('Molecule Size Distribution')
axes[0].axvline(np.median(num_atoms_arr), color='#e74c3c', linestyle='--',
                label=f'Median: {np.median(num_atoms_arr):.0f}')
axes[0].legend()

axes[1].hist(num_edges_arr, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Number of Bonds')
axes[1].set_ylabel('Count')
axes[1].set_title('Bond Count Distribution')

counts = pd.Series(labels_arr_clean).value_counts().sort_index()
axes[2].bar(['Non-penetrating (0)', 'Penetrating (1)'], counts.values,
           color=['#e74c3c', '#3498db'], edgecolor='white')
axes[2].set_ylabel('Count')
axes[2].set_title('BBB Penetration Label Distribution')
for i, v in enumerate(counts.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nAtoms per molecule: mean={num_atoms_arr.mean():.1f}, "
      f"min={num_atoms_arr.min()}, max={num_atoms_arr.max()}")
print(f"Positive rate: {labels_arr_clean.mean():.3f}")

In [ ]:
# ── Visualize a few molecules using RDKit ─────────────────────────────────
from rdkit import Chem
from rdkit.Chem import Draw

smiles_list = [dataset[i].smiles for i in range(len(dataset)) if dataset[i].smiles]
labels_list = [dataset[i].y.squeeze().item() for i in range(len(dataset))]

# Select 4 positive and 4 negative examples
pos_idx = [i for i, l in enumerate(labels_list) if l == 1 and not np.isnan(l)][:4]
neg_idx = [i for i, l in enumerate(labels_list) if l == 0 and not np.isnan(l)][:4]

mols = []
legends = []
for i in pos_idx:
    m = Chem.MolFromSmiles(smiles_list[i])
    if m:
        mols.append(m)
        legends.append(f"BBB+ ({num_atoms_list[i]} atoms)")
for i in neg_idx:
    m = Chem.MolFromSmiles(smiles_list[i])
    if m:
        mols.append(m)
        legends.append(f"BBB- ({num_atoms_list[i]} atoms)")

img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 250), legends=legends)
img

### 🤔 Reflection 2.1 — Understanding Molecular Graph Representations

1. Each node has a 9-dimensional feature vector. These typically encode:
   atomic number, degree, formal charge, number of Hs, hybridization, aromaticity, etc.
   Why might encoding these as *categorical one-hot* features (rather than raw integers)
   be important for learning?

2. The edge_index has shape `[2, 2*num_bonds]` — each bond appears twice (once per
   direction). Why does PyG store undirected edges this way? How does this relate to
   the message-passing paradigm we'll see in Part 4?

3. The dataset is **imbalanced** (~75% positive). How does this affect our choice
   of evaluation metric? Why might accuracy be misleading?

4. The molecules vary in size from ~5 to ~130 atoms. What challenges does this
   create for both classical ML approaches and for neural networks?

In [ ]:
# ══ SOLUTION — Reflection 2.1 ═══════════════════════════════════════════════
# 1. Raw integers (e.g., atomic_number=6 for carbon) impose an arbitrary ordering
#    (the model might learn that oxygen at 8 is "greater" than carbon at 6). One-hot
#    encoding treats each element as a distinct category without ordinal assumptions.
#    This matters because chemical properties are NOT monotonic in atomic number.
#
# 2. In message passing, each node aggregates information from its neighbors.
#    Storing edges as (u->v) AND (v->u) means that when we "send messages along edges,"
#    both endpoints receive messages from the other. This is the standard sparse
#    representation in PyG — it avoids materializing the full n*n adjacency matrix.
#
# 3. With 75% positive, a trivial classifier predicting all-positive gets 75% accuracy.
#    We should use AUROC (threshold-free, handles imbalance) or AUPRC (especially
#    meaningful when we care about the minority class). We'll use AUROC as our primary
#    metric, consistent with the MoleculeNet benchmark.
#
# 4. For classical ML: we need to extract fixed-length features regardless of molecule
#    size, which loses structural information. For neural networks: we can't simply
#    stack variable-size tensors into a batch. GNNs handle this via graph-level pooling
#    (aggregating all node embeddings into a fixed-size graph embedding).
print("See comments above for solution.")

---
## Part 3 — Baseline: Graph Tabularization

Before building a GNN, we establish a baseline by converting each graph into a
fixed-length feature vector — this is the "tabularization" approach. If a simple
feature extraction + logistic regression works well, a GNN may not be needed.

### Two Approaches to Tabularize Molecular Graphs

**Approach A — Hand-crafted graph statistics**: Extract structural features like
number of atoms, number of edges, degree statistics, clustering coefficient, etc.
This loses most chemical information but captures basic topology.

**Approach B — Molecular fingerprints**: Use domain-specific tools (RDKit) to
compute Morgan/ECFP fingerprints — fixed-length binary vectors that encode the
presence of molecular substructures. This is the standard baseline in cheminformatics.

In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────────
# Implement graph-level feature extraction to create a tabular representation.
#
# For EACH molecule in the dataset, compute these features:
#   1. num_atoms       — number of nodes
#   2. num_bonds       — number of edges (undirected)
#   3. mean_degree     — average node degree
#   4. max_degree      — maximum node degree
#   5. density         — 2*num_bonds / (num_atoms * (num_atoms - 1))  [edge density]
#   6. mean_node_feat  — mean of each node feature column (9 values)
#   7. std_node_feat   — std of each node feature column (9 values)
#
# This gives a feature vector of length 5 + 9 + 9 = 23 per molecule.

def extract_graph_features(data):
    x = data.x.numpy()              # node features: (num_atoms, 9)
    edge_index = data.edge_index     # shape: (2, num_directed_edges)

    num_atoms = ???                  # TODO
    num_bonds = ???                  # TODO (remember edges are stored twice)
    degrees = ???                    # TODO: compute degree of each node
    mean_degree = ???                # TODO
    max_degree = ???                 # TODO
    density = ???                    # TODO: edge density

    mean_node_feat = ???             # TODO: mean of each feature column
    std_node_feat = ???              # TODO: std of each feature column

    features = np.concatenate([
        [num_atoms, num_bonds, mean_degree, max_degree, density],
        mean_node_feat,
        std_node_feat
    ])
    return features

# Test on first molecule
test_feat = extract_graph_features(dataset[0])
print(f"Feature vector shape: {test_feat.shape}")
print(f"Features: {test_feat.round(3)}")

In [ ]:
# ══ SOLUTION — extract_graph_features ════════════════════════════════════════
def extract_graph_features(data):
    x = data.x.numpy()
    edge_index = data.edge_index

    num_atoms = x.shape[0]
    num_bonds = edge_index.shape[1] // 2
    degrees = degree(edge_index[0], num_nodes=num_atoms).numpy()
    mean_degree = degrees.mean()
    max_degree = degrees.max()
    if num_atoms > 1:
        density = 2 * num_bonds / (num_atoms * (num_atoms - 1))
    else:
        density = 0.0

    mean_node_feat = x.mean(axis=0)
    std_node_feat = x.std(axis=0)

    features = np.concatenate([
        [num_atoms, num_bonds, mean_degree, max_degree, density],
        mean_node_feat,
        std_node_feat
    ])
    return features

test_feat = extract_graph_features(dataset[0])
print(f"Feature vector shape: {test_feat.shape}")
print(f"Features: {test_feat.round(3)}")

In [ ]:
# ── Build tabular dataset from ALL molecules ──────────────────────────────
print("Extracting graph features for all molecules...")
all_features = []
all_labels = []
valid_indices = []

for i, data in enumerate(dataset):
    label = data.y.squeeze().item()
    if np.isnan(label):
        continue
    all_features.append(extract_graph_features(data))
    all_labels.append(int(label))
    valid_indices.append(i)

X_tab = np.array(all_features)
y_tab = np.array(all_labels)
print(f"Tabular dataset: {X_tab.shape[0]} molecules x {X_tab.shape[1]} features")
print(f"Positive rate: {y_tab.mean():.3f}")

# Train/val/test split (80/10/10)
X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X_tab, y_tab, valid_indices, test_size=0.2, random_state=42, stratify=y_tab)
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"\nSplit sizes: train={len(y_train)}, val={len(y_val)}, test={len(y_test)}")
print(f"Train positive rate: {y_train.mean():.3f}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
# ── Approach A: Classical ML on hand-crafted features ─────────────────────
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
}

baseline_results = {}
for name, model in baseline_models.items():
    model.fit(X_train_s, y_train)
    val_proba = model.predict_proba(X_val_s)[:, 1]
    val_auc = roc_auc_score(y_val, val_proba)
    baseline_results[name] = {'model': model, 'val_auc': val_auc, 'val_proba': val_proba}
    print(f"{name:25s}  Val AUROC = {val_auc:.4f}")

print("\n(These are baselines using simple graph statistics — no structural info!)")

In [ ]:
# ── Approach B: Molecular fingerprints (Morgan/ECFP) ──────────────────────
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_fingerprint(smiles, radius=2, nbits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    return np.array(fp)

print("Computing Morgan fingerprints...")

def build_fp_matrix(indices):
    fps, valid = [], []
    for i in indices:
        smiles = dataset[i].smiles
        fp = smiles_to_fingerprint(smiles)
        if fp is not None:
            fps.append(fp)
            valid.append(True)
        else:
            valid.append(False)
    return np.array(fps), np.array(valid)

X_train_fp, mask_train = build_fp_matrix(idx_train)
X_val_fp, mask_val = build_fp_matrix(idx_val)
X_test_fp, mask_test = build_fp_matrix(idx_test)

y_train_fp = y_train[mask_train]
y_val_fp = y_val[mask_val]
y_test_fp = y_test[mask_test]

print(f"Fingerprint dimension: {X_train_fp.shape[1]}")
print(f"Train: {len(y_train_fp)}, Val: {len(y_val_fp)}, Test: {len(y_test_fp)}")

fp_models = {
    'LR + Fingerprints': LogisticRegression(max_iter=1000, random_state=42),
    'RF + Fingerprints': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
    'GBT + Fingerprints': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
}

fp_results = {}
for name, model in fp_models.items():
    model.fit(X_train_fp, y_train_fp)
    val_proba = model.predict_proba(X_val_fp)[:, 1]
    val_auc = roc_auc_score(y_val_fp, val_proba)
    fp_results[name] = {'model': model, 'val_auc': val_auc, 'val_proba': val_proba}
    print(f"{name:25s}  Val AUROC = {val_auc:.4f}")

### 🤔 Reflection 3.1 — Tabularization Trade-offs

1. Compare the AUROC of hand-crafted graph features vs. Morgan fingerprints.
   Why do fingerprints perform better? What chemical information do they capture
   that raw graph statistics miss?

2. Morgan fingerprints (ECFP) encode circular substructures of radius $r$ around
   each atom. This is conceptually similar to what operation in GNNs?
   (We'll see this in Part 4.)

3. What information is lost when we tabularize a graph? Give a specific example of
   two molecules that would have identical hand-crafted features but different properties.

4. In a clinical drug-screening pipeline, what are the advantages of the
   fingerprint baseline? When might its simplicity be preferred over a complex GNN?

In [ ]:
# ══ SOLUTION — Reflection 3.1 ═══════════════════════════════════════════════
# 1. Fingerprints dramatically outperform hand-crafted features because they encode
#    LOCAL chemical substructure patterns (e.g., "a nitrogen bonded to two carbons
#    with an aromatic ring nearby"). Graph statistics like mean degree and density
#    only capture global topology — they can't distinguish between chemically
#    very different molecules that happen to have similar sizes.
#
# 2. Morgan fingerprints of radius r enumerate all substructures within r bonds of
#    each atom, then hash them into a fixed-length vector. This is essentially the
#    same operation as r rounds of message passing in a GNN! The key difference is
#    that GNNs learn WHAT to extract, while fingerprints use a fixed hashing scheme.
#    This is why GNNs are sometimes called "learnable fingerprints."
#
# 3. Two molecules with the same number of atoms, bonds, and degree distribution
#    but different connectivity (graph isomers) would have identical hand-crafted
#    features. Example: 1,2-dimethylbenzene vs 1,4-dimethylbenzene (ortho- vs para-
#    xylene) — same atom counts and degree stats, different biological properties.
#
# 4. Advantages: (a) Very fast to compute and train — seconds vs. minutes/hours for
#    GNNs. (b) Highly interpretable — you can identify which substructures drive
#    predictions. (c) Robust baselines that are hard to beat on small datasets.
#    (d) No GPU needed. Preferred when: dataset is small (<5K), interpretability is
#    critical, or as a rapid screening step before expensive GNN-based analysis.
print("See comments above for solution.")

---
## Part 4 — Understanding Message Passing

The core idea behind GNNs is **message passing**: each node updates its representation
by aggregating information from its neighbors. After $K$ rounds of message passing,
each node's embedding encodes information about its $K$-hop neighborhood.

### The Message Passing Framework

At each layer $k$, node $v$ updates its embedding $\mathbf{h}_v^{(k)}$ via:

$$\mathbf{h}_v^{(k)} = \text{UPDATE}\left(\mathbf{h}_v^{(k-1)},\; \text{AGGREGATE}\left(\left\{\mathbf{h}_u^{(k-1)} : u \in \mathcal{N}(v)\right\}\right)\right)$$

where $\mathcal{N}(v)$ is the set of neighbors of node $v$.

Different GNN architectures differ in their choice of AGGREGATE and UPDATE:

| Architecture | AGGREGATE | UPDATE |
|---|---|---|
| **GCN** | Normalized mean of neighbors | Linear + activation |
| **GraphSAGE** | Mean, max, or LSTM over neighbors | Concatenate + linear |
| **GIN** | Sum of neighbors | MLP |
| **GAT** | Attention-weighted sum | Linear + activation |

We'll implement the simplest version — a **Graph Convolutional Network (GCN)** layer — by hand.

In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────────
# Implement ONE layer of GCN-style message passing.
#
# The GCN update rule (simplified, without normalization) is:
#   H^(k) = ReLU( A_tilde @ H^(k-1) @ W )
#
# where:
#   A_tilde = A + I  (adjacency matrix with self-loops)
#   H^(k-1) is the node feature matrix at layer k-1
#   W is a learnable weight matrix
#
# Use the first molecule in the dataset.

def gcn_layer_manual(A, H, W):
    # Step 1: Add self-loops (A_tilde = A + I)
    A_tilde = ???  # TODO

    # Step 2: Aggregate neighbor features (A_tilde @ H)
    agg = ???      # TODO

    # Step 3: Linear transform (agg @ W)
    out = ???      # TODO

    # Step 4: Apply ReLU activation
    H_new = ???    # TODO

    return H_new

# ── Test on first molecule ────────────────────────────────────────────────
mol = dataset[0]
A = to_dense_adj(mol.edge_index, max_num_nodes=mol.x.shape[0]).squeeze(0).numpy()
H = mol.x.numpy()  # shape: (num_atoms, 9)

np.random.seed(42)
W = np.random.randn(9, 16) * 0.1

H_new = gcn_layer_manual(A, H, W)
print(f"Input features shape:  {H.shape}")
print(f"Output features shape: {H_new.shape}")
print(f"Output (first 3 nodes, first 5 dims):\n{H_new[:3, :5].round(4)}")

In [ ]:
# ══ SOLUTION — gcn_layer_manual ══════════════════════════════════════════════
def gcn_layer_manual(A, H, W):
    n = A.shape[0]
    # Step 1: Add self-loops
    A_tilde = A + np.eye(n)
    # Step 2: Aggregate neighbor features
    agg = A_tilde @ H
    # Step 3: Linear transform
    out = agg @ W
    # Step 4: ReLU
    H_new = np.maximum(0, out)
    return H_new

mol = dataset[0]
A = to_dense_adj(mol.edge_index, max_num_nodes=mol.x.shape[0]).squeeze(0).numpy()
H = mol.x.numpy()
np.random.seed(42)
W = np.random.randn(9, 16) * 0.1
H_new = gcn_layer_manual(A, H, W)
print(f"Input features shape:  {H.shape}")
print(f"Output features shape: {H_new.shape}")
print(f"Output (first 3 nodes, first 5 dims):\n{H_new[:3, :5].round(4)}")

In [ ]:
# ── Visualize message passing: before and after ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
H_2d = pca.fit_transform(H)
H_new_2d = pca.fit_transform(H_new)

# Build networkx graph for layout
G_mol = nx.Graph()
edge_list = mol.edge_index.numpy().T
for e in edge_list:
    G_mol.add_edge(int(e[0]), int(e[1]))
pos = nx.spring_layout(G_mol, seed=42)

nx.draw(G_mol, pos, ax=axes[0], node_color='#3498db', node_size=300,
        edge_color='#bbb', width=1.5, with_labels=True, font_size=8, font_color='white')
axes[0].set_title("Graph Structure", fontsize=12)

node_colors_before = H_2d[:, 0]
nx.draw(G_mol, pos, ax=axes[1], node_color=node_colors_before, node_size=300,
        cmap='coolwarm', edge_color='#bbb', width=1.5, with_labels=True, font_size=8,
        font_color='white')
axes[1].set_title("Node Features: Before MP", fontsize=12)

node_colors_after = H_new_2d[:, 0]
nx.draw(G_mol, pos, ax=axes[2], node_color=node_colors_after, node_size=300,
        cmap='coolwarm', edge_color='#bbb', width=1.5, with_labels=True, font_size=8,
        font_color='white')
axes[2].set_title("Node Features: After 1 MP Layer", fontsize=12)

plt.suptitle("Message Passing Smooths Features Across Neighbors", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 🤔 Reflection 4.1 — Message Passing Mechanics

1. After one round of message passing, each node's embedding contains information
   about itself and its **1-hop neighbors**. After $K$ rounds, each node "sees"
   its $K$-hop neighborhood. How does this relate to the *receptive field* concept
   in CNNs?

2. In the GCN update $\tilde{A} H W$, the multiplication $\tilde{A} H$ is essentially
   averaging each node's features with its neighbors' features. What happens if we
   stack many GCN layers — what problem might arise? (Hint: think about what happens
   when you repeatedly average signals.)

3. We added self-loops ($\tilde{A} = A + I$) before message passing. What would
   happen without self-loops? Why is this important?

4. The GIN (Graph Isomorphism Network) uses **sum** aggregation instead of mean.
   Mathematically, why is sum aggregation strictly more expressive than mean?
   (Hint: consider two nodes where one has 2 neighbors with feature [1] and another
   has 4 neighbors with feature [1]. What does mean vs. sum tell you?)

In [ ]:
# ══ SOLUTION — Reflection 4.1 ═══════════════════════════════════════════════
# 1. This is exactly analogous to the receptive field in CNNs. In a CNN, a neuron
#    in layer K "sees" a spatial region of the input determined by K convolutions.
#    In a GNN, K message-passing layers give each node a "receptive field" of K-hop
#    neighborhoods. Just as deeper CNNs have larger receptive fields, deeper GNNs
#    aggregate information from more distant nodes.
#
# 2. This is the "over-smoothing" problem. After many layers of averaging, all node
#    embeddings converge to the same value (the graph's mean feature). This is why
#    GNNs typically use only 2-5 layers, unlike CNNs which can be very deep.
#    Techniques like residual connections and jumping knowledge help mitigate this.
#
# 3. Without self-loops, each node's new embedding is the aggregate of ONLY its
#    neighbors' features — the node's own current features are lost! Self-loops
#    ensure the node's own representation is included in the aggregation, similar
#    to a residual connection.
#
# 4. Mean aggregation maps {1, 1} and {1, 1, 1, 1} to the same output (mean = 1),
#    making these neighborhoods indistinguishable. Sum aggregation gives 2 vs 4,
#    preserving the multiset structure. The WL isomorphism test (which GIN is based
#    on) requires injective aggregation, and sum over multisets is injective while
#    mean is not. This is why GIN is provably more expressive than GCN.
print("See comments above for solution.")

---
## Part 5 — Training a Graph Neural Network

Now we train a full GNN using PyTorch Geometric. We'll implement two architectures:

1. **GCN** (Graph Convolutional Network) — the simplest GNN
2. **GIN** (Graph Isomorphism Network) — provably more expressive

Both follow the same pipeline:
1. Multiple message-passing layers produce node embeddings
2. A **readout** (pooling) function aggregates node embeddings into a single graph embedding
3. A classification head maps the graph embedding to a prediction

### Graph-Level Readout

Since we need one prediction per *graph* (not per node), we need to aggregate all
node embeddings into a fixed-size graph representation:
- **Mean pooling**: $\mathbf{h}_G = \frac{1}{|V|} \sum_{v \in V} \mathbf{h}_v$
- **Sum pooling**: $\mathbf{h}_G = \sum_{v \in V} \mathbf{h}_v$
- **Max pooling**: $\mathbf{h}_G = \max_{v \in V} \mathbf{h}_v$ (element-wise)

In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────────
# Implement a GCN for graph classification.
#
# Architecture:
#   Input node features (9-dim)
#   -> GCNConv(9, hidden_dim) -> ReLU -> Dropout
#   -> GCNConv(hidden_dim, hidden_dim) -> ReLU -> Dropout
#   -> Global Mean Pool (aggregate nodes -> graph embedding)
#   -> Linear(hidden_dim, 1) -> Sigmoid
#
# Use torch_geometric.nn.GCNConv for the graph convolution layers.

class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = ???       # TODO: GCNConv input_dim -> hidden_dim
        self.conv2 = ???       # TODO: GCNConv hidden_dim -> hidden_dim
        self.classifier = ???  # TODO: Linear hidden_dim -> 1
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        # Layer 1: GCNConv -> ReLU -> Dropout
        h = ???              # TODO

        # Layer 2: GCNConv -> ReLU -> Dropout
        h = ???              # TODO

        # Readout: aggregate node embeddings to graph-level
        h_graph = ???        # TODO: use global_mean_pool(h, batch)

        # Classify
        out = ???            # TODO: linear -> sigmoid -> squeeze
        return out

model_gcn = GCN(input_dim=dataset.num_node_features, hidden_dim=64)
print(model_gcn)
print(f"\nTotal parameters: {sum(p.numel() for p in model_gcn.parameters()):,}")

In [ ]:
# ══ SOLUTION — GCN ═══════════════════════════════════════════════════════════
class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h_graph = global_mean_pool(h, batch)

        out = torch.sigmoid(self.classifier(h_graph)).squeeze(-1)
        return out

model_gcn = GCN(input_dim=dataset.num_node_features, hidden_dim=64)
print(model_gcn)
print(f"\nTotal parameters: {sum(p.numel() for p in model_gcn.parameters()):,}")

In [ ]:
# ── GIN: A more expressive GNN (provided) ─────────────────────────────────
# The GIN uses an MLP instead of a linear layer after aggregation,
# and uses SUM pooling instead of MEAN for both message passing and readout.

class GIN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        mlp1 = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(),
                             nn.Linear(hidden_dim, hidden_dim))
        mlp2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                             nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h_graph = global_add_pool(h, batch)  # SUM pooling
        out = torch.sigmoid(self.classifier(h_graph)).squeeze(-1)
        return out

model_gin = GIN(input_dim=dataset.num_node_features, hidden_dim=64)
print(model_gin)
print(f"\nGIN parameters: {sum(p.numel() for p in model_gin.parameters()):,}")

In [ ]:
# ── Prepare PyG DataLoaders ────────────────────────────────────────────────
# We use the same train/val/test split indices from Part 3

valid_dataset = [dataset[i] for i in valid_indices]

idx_map = {orig: new for new, orig in enumerate(valid_indices)}
train_data = [valid_dataset[idx_map[i]] for i in idx_train]
val_data = [valid_dataset[idx_map[i]] for i in idx_val]
test_data = [valid_dataset[idx_map[i]] for i in idx_test]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, "
      f"Test batches: {len(test_loader)}")

batch = next(iter(train_loader))
print(f"\nBatch object: {batch}")
print(f"  Batch node features shape: {batch.x.shape}")
print(f"  Batch edge_index shape: {batch.edge_index.shape}")
print(f"  Batch vector shape: {batch.batch.shape}")
print(f"  Batch labels shape: {batch.y.shape}")
print(f"  Number of graphs in batch: {batch.num_graphs}")

In [ ]:
# ── Training and evaluation functions ─────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x.float(), batch.edge_index, batch.batch)
        labels = batch.y.squeeze().float()
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    criterion = nn.BCELoss()
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x.float(), batch.edge_index, batch.batch)
        labels = batch.y.squeeze().float()
        total_loss += criterion(out, labels).item() * batch.num_graphs
        all_preds.append(out.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc = roc_auc_score(labels, preds)
    loss = total_loss / len(loader.dataset)
    return auc, loss, preds, labels

def train_gnn(model, train_loader, val_loader, lr=1e-3, epochs=80, patience=15):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCELoss()

    history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}
    best_val_auc = 0
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        train_auc, _, _, _ = evaluate(model, train_loader)
        val_auc, val_loss, _, _ = evaluate(model, val_loader)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_auc'].append(train_auc)
        history['val_auc'].append(val_auc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | "
                  f"Val AUC: {val_auc:.4f} | Best: {best_val_auc:.4f}")

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, history

In [ ]:
# ── Train GCN ─────────────────────────────────────────────────────────────
print("=" * 60)
print("Training GCN")
print("=" * 60)
gcn_model = GCN(input_dim=dataset.num_node_features, hidden_dim=64, dropout=0.3)
gcn_model, gcn_history = train_gnn(gcn_model, train_loader, val_loader, lr=1e-3, epochs=80)

print("\n" + "=" * 60)
print("Training GIN")
print("=" * 60)
gin_model = GIN(input_dim=dataset.num_node_features, hidden_dim=64, dropout=0.3)
gin_model, gin_history = train_gnn(gin_model, train_loader, val_loader, lr=1e-3, epochs=80)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for idx, (name, hist) in enumerate([('GCN', gcn_history), ('GIN', gin_history)]):
    axes[0, idx].plot(hist['train_loss'], label='Train', color='#3498db')
    axes[0, idx].plot(hist['val_loss'], label='Val', color='#e74c3c')
    axes[0, idx].set_xlabel('Epoch'); axes[0, idx].set_ylabel('BCE Loss')
    axes[0, idx].set_title(f'{name} — Loss Curves'); axes[0, idx].legend()

    axes[1, idx].plot(hist['train_auc'], label='Train', color='#3498db')
    axes[1, idx].plot(hist['val_auc'], label='Val', color='#e74c3c')
    axes[1, idx].set_xlabel('Epoch'); axes[1, idx].set_ylabel('AUROC')
    axes[1, idx].set_title(f'{name} — AUROC Curves'); axes[1, idx].legend()

plt.tight_layout()
plt.show()

### 🤔 Reflection 5.1 — GNN Architecture and Training

1. Compare the parameter counts of GCN vs. GIN. Why does GIN have more parameters?
   What is the purpose of the MLP inside each GIN layer?

2. Look at the training curves. Do you see signs of overfitting? How does the gap
   between train and validation AUROC compare between GCN and GIN?

3. We used `global_mean_pool` for GCN and `global_add_pool` for GIN.
   For a molecule with 10 atoms vs. one with 50 atoms, how do these pooling
   strategies differ? Which preserves information about molecule *size*?

4. Our GNN has only 2 layers. For a molecule with 20 atoms in a chain, what is
   the maximum path length between two atoms whose information can interact?
   When might we want more layers, and what is the trade-off?

In [ ]:
# ══ SOLUTION — Reflection 5.1 ═══════════════════════════════════════════════
# 1. GIN uses a 2-layer MLP (linear -> ReLU -> linear) as the UPDATE function in
#    each message-passing layer, while GCN uses a single linear transform. This
#    extra nonlinearity allows GIN to learn more complex node-level transformations.
#    The MLP acts as a universal function approximator within each layer, making
#    the aggregation injective (provably as powerful as the WL test).
#
# 2. Both models likely show some overfitting (train AUC > val AUC), but GIN may
#    overfit more due to its higher capacity. The dataset is small (~2K molecules),
#    making overfitting a persistent challenge. Dropout and early stopping help but
#    can't fully compensate for limited data.
#
# 3. Mean pooling divides by |V|, so a 50-atom molecule's graph embedding has the
#    same scale as a 10-atom molecule's. Sum pooling does NOT normalize, so larger
#    molecules tend to have larger embedding norms. Sum pooling preserves size info,
#    which can be important (larger molecules often have different properties). This
#    is why GIN recommends sum pooling — it's more expressive.
#
# 4. With 2 layers, each node's embedding captures its 2-hop neighborhood. In a
#    20-atom chain, atoms at positions 1 and 20 are 19 hops apart — they cannot
#    directly influence each other. We'd need ~10 layers for full coverage, but
#    over-smoothing would degrade representations long before that. Solutions include
#    virtual nodes (connected to all atoms), jumping knowledge (concatenating all
#    layer outputs), or positional encodings.
print("See comments above for solution.")

---
## Part 6 — Model Comparison and Final Test Evaluation

We now compare all approaches — tabularization baselines and GNNs — on the held-out
test set. As in the earlier labs, **this is the first time we touch the test set**.
All model selection and hyperparameter tuning was done using the validation set.

In [ ]:
# ── Collect validation results for model selection ────────────────────────
print("=== Validation Set Results (for model selection) ===\n")

all_results = {}

for name, res in baseline_results.items():
    all_results[name] = res['val_auc']
    print(f"  {name:30s}  Val AUROC = {res['val_auc']:.4f}")

for name, res in fp_results.items():
    all_results[name] = res['val_auc']
    print(f"  {name:30s}  Val AUROC = {res['val_auc']:.4f}")

gcn_val_auc, _, _, _ = evaluate(gcn_model, val_loader)
gin_val_auc, _, _, _ = evaluate(gin_model, val_loader)
all_results['GCN'] = gcn_val_auc
all_results['GIN'] = gin_val_auc
print(f"  {'GCN':30s}  Val AUROC = {gcn_val_auc:.4f}")
print(f"  {'GIN':30s}  Val AUROC = {gin_val_auc:.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
names = list(all_results.keys())
aucs = list(all_results.values())
colors = ['#bdc3c7']*3 + ['#95a5a6']*3 + ['#3498db', '#e74c3c']
bars = ax.barh(names, aucs, color=colors, edgecolor='white')
ax.set_xlabel('Validation AUROC')
ax.set_title('Model Comparison — Validation AUROC')
ax.set_xlim(0.5, 1.0)
for bar, v in zip(bars, aucs):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.4f}',
            va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final test evaluation ─────────────────────────────────────────────────
print("=== FINAL TEST SET RESULTS ===")
print("(All models selected on validation set; test set never seen before)\n")

best_fp_name = max(fp_results, key=lambda k: fp_results[k]['val_auc'])
best_fp_model = fp_results[best_fp_name]['model']
fp_test_proba = best_fp_model.predict_proba(X_test_fp)[:, 1]
fp_test_auc = roc_auc_score(y_test_fp, fp_test_proba)

best_bl_name = max(baseline_results, key=lambda k: baseline_results[k]['val_auc'])
best_bl_model = baseline_results[best_bl_name]['model']
bl_test_proba = best_bl_model.predict_proba(X_test_s)[:, 1]
bl_test_auc = roc_auc_score(y_test, bl_test_proba)

gcn_test_auc, _, gcn_test_preds, gcn_test_labels = evaluate(gcn_model, test_loader)
gin_test_auc, _, gin_test_preds, gin_test_labels = evaluate(gin_model, test_loader)

test_summary = {
    f'Best Graph Stats ({best_bl_name})': bl_test_auc,
    f'Best Fingerprints ({best_fp_name})': fp_test_auc,
    'GCN': gcn_test_auc,
    'GIN': gin_test_auc,
}

for name, auc_val in test_summary.items():
    print(f"  {name:45s}  Test AUROC = {auc_val:.4f}")

# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))

fpr, tpr, _ = roc_curve(y_test_fp, fp_test_proba)
ax.plot(fpr, tpr, label=f'Best Fingerprints (AUC={fp_test_auc:.3f})', color='#95a5a6', lw=2)

fpr, tpr, _ = roc_curve(y_test, bl_test_proba)
ax.plot(fpr, tpr, label=f'Best Graph Stats (AUC={bl_test_auc:.3f})', color='#bdc3c7', lw=2)

fpr, tpr, _ = roc_curve(gcn_test_labels, gcn_test_preds)
ax.plot(fpr, tpr, label=f'GCN (AUC={gcn_test_auc:.3f})', color='#3498db', lw=2)

fpr, tpr, _ = roc_curve(gin_test_labels, gin_test_preds)
ax.plot(fpr, tpr, label=f'GIN (AUC={gin_test_auc:.3f})', color='#e74c3c', lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 🤔 Reflection 6.1 — Comparing Approaches

1. How do the GNNs compare to the fingerprint baselines? Is the improvement
   (if any) large enough to justify the additional complexity and compute cost?

2. The gap between hand-crafted graph statistics and fingerprints is typically
   larger than the gap between fingerprints and GNNs. Why? What does this tell
   us about where the "information bottleneck" lies in molecular prediction?

3. If you were building a drug screening pipeline and needed to classify 10 million
   molecules, which approach would you use first? How would you combine the approaches?

4. In what scenarios might GNNs provide a larger advantage over fingerprints?
   (Hint: think about 3D structure, larger molecules, or tasks beyond classification.)

In [ ]:
# ══ SOLUTION — Reflection 6.1 ═══════════════════════════════════════════════
# 1. On BBBP, GNNs and fingerprints tend to perform comparably. The improvement
#    from GNNs (if any) is often small (1-3% AUROC) — not always enough to justify
#    the compute overhead for a production screening pipeline. This is a well-known
#    finding: molecular fingerprints are extremely strong baselines.
#
# 2. The biggest information loss happens when going from structural representation
#    to flat statistics (losing all chemical substructure). Fingerprints preserve
#    local chemical neighborhoods via hashing, which captures most of the predictive
#    signal. GNNs learn the same kind of patterns but with learned (instead of fixed)
#    aggregation functions. The marginal gain from learning vs. hashing is small when
#    the fingerprint space is already well-designed by decades of cheminformatics.
#
# 3. For 10M molecules: (a) use fingerprints for rapid initial screening (seconds
#    per molecule on CPU), (b) apply GNN to the top ~100K candidates for refined
#    ranking, (c) run expensive physics simulations on the top ~1K. This cascade
#    architecture is standard in virtual screening pipelines.
#
# 4. GNNs provide larger advantages for: (a) 3D structure-aware tasks (using 3D
#    coordinates as features), (b) tasks requiring global molecular properties that
#    go beyond local substructure patterns, (c) multi-task learning where shared
#    representations across tasks are valuable, (d) learning on novel chemical
#    spaces where standard fingerprints may not have relevant patterns.
print("See comments above for solution.")

---
## Part 7 — Extensions: What Can You Do From Here?

If you have extra time or want to explore further, try any of these extensions.
They are optional.

In [ ]:
# ── Extension 1: Vary GNN depth and observe over-smoothing ────────────────
# Train GCNs with 1, 2, 3, 4, 6 layers and plot val AUROC vs depth.

depths = [1, 2, 3, 4, 6]
depth_results = {}

class GCN_Variable(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        h = x
        for conv in self.convs:
            h = F.relu(conv(h, edge_index))
            h = F.dropout(h, p=self.dropout, training=self.training)
        h_graph = global_mean_pool(h, batch)
        return torch.sigmoid(self.classifier(h_graph)).squeeze(-1)

for d in depths:
    print(f"\n--- Depth {d} ---")
    m = GCN_Variable(dataset.num_node_features, hidden_dim=64, num_layers=d)
    m, hist = train_gnn(m, train_loader, val_loader, lr=1e-3, epochs=60, patience=10)
    val_auc, _, _, _ = evaluate(m, val_loader)
    depth_results[d] = val_auc
    print(f"  Val AUROC: {val_auc:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(list(depth_results.keys()), list(depth_results.values()),
         'o-', color='#3498db', lw=2)
plt.xlabel('Number of GCN Layers')
plt.ylabel('Validation AUROC')
plt.title('Over-smoothing: GNN Performance vs Depth')
plt.xticks(depths)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🧠 Final Reflection — Graph Data in the ML Landscape

Now that you've worked through graph-structured data alongside the tabular data from
earlier labs, answer these synthesis questions:

1. **Representation matters**: We saw that hand-crafted graph statistics $\ll$ fingerprints $\approx$ GNNs.
   What is the general lesson here about the importance of *how* you represent data
   vs. *which* model you use? How does this relate to the "feature engineering vs.
   feature learning" debate?

2. **Inductive bias**: GNNs build in the assumption that graph structure matters —
   neighbors influence each other. CNNs assume spatial locality. Transformers assume
   everything can attend to everything. How does the *inductive bias* of each
   architecture match or mismatch with different data types in healthcare?

3. **When is graph structure essential?** For the BBBP task, fingerprints
   (which hash the graph) performed nearly as well as GNNs (which learn from it).
   Can you think of a biomedical problem where the graph structure would be much
   more important — where you truly *need* a GNN? (Hint: think protein-protein
   interactions, drug-drug interactions, or patient similarity networks.)

4. **Scalability and deployment**: In the earlier labs, all models could be trained
   in seconds on a laptop. The GNN required PyTorch, GPUs, and specialized batching.
   How should the added complexity of graph methods influence your decision to use
   them in a clinical tool?

5. **Looking ahead**: This lab used 2D molecular graphs (connectivity only). Modern
   molecular ML increasingly uses 3D coordinates (positions of atoms in space).
   What additional information does 3D structure provide, and why is it especially
   important for drug-receptor binding prediction?

In [ ]:
# ══ SOLUTION — Final Reflection ══════════════════════════════════════════════
# 1. Representation dominates model choice for most problems. The jump from simple
#    statistics to fingerprints (preserving local structure) was much larger than
#    from fingerprints to GNNs. This echoes the broader ML lesson: good features
#    (whether handcrafted or learned) matter more than algorithmic sophistication.
#    Feature engineering = human designs representations; feature learning = model
#    discovers them. GNNs are "learnable fingerprints" — but the fixed fingerprints
#    were already very well designed by decades of cheminformatics research.
#
# 2. GNNs: molecular graphs, protein structures, knowledge graphs — any data with
#    explicit relational structure. CNNs: medical images (X-rays, pathology), where
#    spatial locality matters. Transformers: clinical text (EHR notes, radiology
#    reports), where long-range dependencies matter. The best architecture matches
#    the data's natural structure. Using the wrong inductive bias (e.g., treating
#    graphs as flat feature vectors) wastes information.
#
# 3. Drug-drug interaction prediction: you must know WHICH atoms in drug A interact
#    with WHICH atoms in drug B — this requires graph-level reasoning about molecular
#    substructures that can't be captured by global fingerprints. Protein function
#    prediction from contact maps also requires understanding spatial relationships
#    between residues. Patient similarity networks where the network topology itself
#    (e.g., shared rare diseases) is the signal.
#
# 4. Added complexity increases development time, debugging difficulty, and
#    deployment costs. For clinical deployment, simpler models are preferable unless
#    graph methods provide a clinically meaningful improvement. The decision should
#    be: does the extra 2% AUROC translate to better patient outcomes? If not, use
#    the simpler model. Also consider: reproducibility, interpretability, regulatory
#    approval — all harder with complex deep learning pipelines.
#
# 5. 3D structure provides information about molecular shape, which determines how
#    a drug molecule fits into a protein's binding pocket (like a key in a lock).
#    Two molecules with identical 2D graphs can have different 3D conformations
#    (spatial arrangements) that dramatically affect binding affinity. For drug-
#    receptor binding prediction, 3D-aware models (e.g., SchNet, DimeNet, Equiformer)
#    can capture these interactions, while 2D methods cannot.
print("See comments above for solution.")